# Quick start

This notebook reads the IEEE 14-bus case, imports it to CIM, looks at the `.cimtbl` text the CIM model was built from,
and exports it back to a `.raw` file. It runs from `docs/guide/` in the repository, which holds the case at
`../../tests/cases/IEEE14/IEEE_14_bus.raw`. Install the package first ([Installation](installation.md)).

| Step | Call | Touches files |
|---|---|---|
| Read | `Raw34Reader(path).read_all()` returns a `GraphModel` of RAW records | reads the `.raw` |
| Import | `raw_to_cim(model, name=...)` returns a cim18v17 network | writes `.cimtbl` artifacts to a temporary directory unless you pass `artifact_dir=` |
| Export | `RawExporter().to_model(network)` returns a `GraphModel` of RAW records | none |
| Write | `Raw34Writer().write(model, path)` | writes the `.raw` |

Only the reader and the writer read and write case files. `raw_to_cim` writes its `.cimtbl` text to disk only to build the network from it.

## Read the case

`Raw34Reader` reads a RAW v34 file and returns a `GraphModel`: records keyed by their record class, one class per RAW
table. The IEEE 14-bus case has 14 buses, 11 loads, 5 generators, 17 branches and 3 transformers.

In [1]:
from pathlib import Path

from cimhub_raw.raw34_reader import Raw34Reader

case = Path("../../tests/cases/IEEE14/IEEE_14_bus.raw")
model = Raw34Reader(case).read_all()
for record_class, records in model.graph.items():
    print(f"{record_class.__name__:28} {len(records)}")

CaseIdentificationRecord     1
BusRecord                    14
LoadRecord                   11
FixedShuntRecord             1
GeneratorRecord              5
BranchRecord                 17
TransformerRecord            3
AreaInterchangeRecord        1
ZoneRecord                   1
OwnerRecord                  1


Each record is a dataclass from the LinkML schema, with RAW's own values in RAW's own units. This is bus 1 of the case:
`base_voltage` is BASKV in kV and `voltage_magnitude` is VM in per unit.

In [2]:
import cimhub_raw.schema.raw_v34.raw34_schema as schema

print(model.list_by_class(schema.BusRecord)[0])

BusRecord({
  'bus_number': 1,
  'name': 'BUS 1',
  'base_voltage': 138.0,
  'bus_type': 3,
  'area': 1,
  'zone': 1,
  'owner': 1,
  'voltage_magnitude': 1.06,
  'voltage_angle': 0.0,
  'normal_voltage_magnitude_high_limit': 1.1,
  'normal_voltage_magnitude_low_limit': 0.9,
  'emergency_voltage_magnitude_high_limit': 1.1,
  'emergency_voltage_magnitude_low_limit': 0.9
})


## Import to CIM

`raw_to_cim` maps the records to `.cimtbl` text and has cimtables build the CIM network from it. `name` is the master
document's `Model` and file stem. `artifact_dir` keeps the text; here it goes to a temporary directory.

In [3]:
import tempfile

from cimhub_raw.importer.raw_to_cimtbl import raw_to_cim

work = Path(tempfile.mkdtemp())
network = raw_to_cim(model, name="IEEE14", artifact_dir=work / "cimtbl")

The network is a cim-graph `BusBranchModel`. `network.graph` holds its objects by CIM class. Each of the 14 buses is a
`TopologicalNode`, the 17 branches are `ACLineSegment`s and the 3 transformers are `PowerTransformer`s. The buses are in
11 `Substation`s, because a transformer joins its buses into one substation: buses 4, 7 and 9 share one, and buses 5 and 6
another.

In [4]:
import cimprofiles.cim18.cim18v17 as cim

for cim_class in (cim.TopologicalNode, cim.Substation, cim.ACLineSegment, cim.PowerTransformer, cim.ConformLoad,
                  cim.SynchronousMachine, cim.LinearShuntCompensator):
    print(f"{cim_class.__name__:24} {len(network.graph.get(cim_class, {}))}")

TopologicalNode          14
Substation               11
ACLineSegment            17
PowerTransformer         3
ConformLoad              11
SynchronousMachine       5
LinearShuntCompensator   1


To read a value, use a [CIMHub core getter](https://pnnl-cim-tools.github.io/CIMHub/core/guide/getters/). It converts
CIM's SI value to the unit you name. Load `3_1` is the 94.2 MW, 19.0 MVAr load at bus 3, and bus 3's solved voltage is
VM x BASKV at angle VA.

In [5]:
from cimhub_core.getters.load import get_phase_loads
from cimhub_core.getters.voltage import get_node_voltage

load = next(obj for obj in network.graph[cim.ConformLoad].values() if obj.name == "3_1")
print(get_phase_loads(load, "MW", "MVAr").total)

bus_3 = next(node for node in network.graph[cim.TopologicalNode].values() if node.busNumber == 3)
print(get_node_voltage(bus_3, "kV", "deg"))

PhasePower(p=94.2, q=19.0)
NodeVoltage(v=139.38, angle=-12.725)


## The `.cimtbl` artifacts

`artifact_dir` holds one file per RAW table the case uses, plus the master, which `Import`s the others in dependency
order. The master also holds the system base, the base frequency, the base voltages and the rating-set names.

In [6]:
artifacts = work / "cimtbl"
print(sorted(path.name for path in artifacts.iterdir()))
print()
master = (artifacts / "IEEE14.cimtbl").read_text().splitlines()
print("\n".join(line for line in master if line.startswith(("Object", "Import"))))

['IEEE14.cimtbl', 'area_interchange.cimtbl', 'branch.cimtbl', 'bus.cimtbl', 'fixed_shunt.cimtbl', 'generator.cimtbl', 'load.cimtbl', 'transformer.cimtbl', 'zone.cimtbl']

Object BasePower: mRID=sbase, basePower (MVA)=100.0
Object BaseFrequency: mRID=bf_60hz, frequency (Hz)=60.0
Object GeographicalRegion: mRID=rgn_ieee14, name=IEEE14
Import zone.cimtbl
Import area_interchange.cimtbl
Import bus.cimtbl
Import branch.cimtbl
Import load.cimtbl
Import fixed_shunt.cimtbl
Import generator.cimtbl
Import transformer.cimtbl


Values are copied as the case gave them, and the unit is in the column header. Branch R, X and B are per unit on the
system base, so their headers say `(pu system)`; CIM's own value is converted from them once, using the master's
`BasePower`. A comment says what the importer defaulted or left out.

In [7]:
lines = (artifacts / "branch.cimtbl").read_text().splitlines()
start = next(index for index, line in enumerate(lines) if line.startswith("Table ACLineSegment"))
print("\n".join(line[:118] for line in [lines[start - 2], *lines[start:start + 4]]))

# untranscribed: o1, f1, o2, f2, o3, f3, o4, f4
Table ACLineSegment: mRID, name, bus1, bus2, r (pu system), x (pu system), bch (pu system), length (km), inService, Ba
    ln_1_2_1,   1_2_1,   1,  2,  0.01938, 0.05917, 0.0528, , true, bv_138kv,
    ln_1_5_1,   1_5_1,   1,  5,  0.05403, 0.22304, 0.0492, , true, bv_138kv,
    ln_2_3_1,   2_3_1,   2,  3,  0.04699, 0.19797, 0.0438, , true, bv_138kv,


## Export to a `.raw`

`RawExporter().to_model` reads the CIM network through the core getters and returns a `GraphModel` of RAW records, the same
shape `Raw34Reader.read_all()` returns. `Raw34Writer().write` writes it.

In [8]:
from cimhub_raw.exporter.cim_to_raw import RawExporter
from cimhub_raw.raw34_writer import Raw34Writer

exported = RawExporter().to_model(network)
for record_class, records in exported.graph.items():
    print(f"{record_class.__name__:28} {len(records)}")

written = work / "out" / "IEEE14.raw"
written.parent.mkdir()
Raw34Writer().write(exported, written)
print()
print(sorted(path.name for path in written.parent.iterdir()))

CaseIdentificationRecord     1
ZoneRecord                   1
AreaInterchangeRecord        1
BusRecord                    14
BranchRecord                 17
LoadRecord                   11
FixedShuntRecord             1
GeneratorRecord              5
TransformerRecord            3

['IEEE14.raw']


The file starts with the case identification line (IC, SBASE, REV, XFRRAT, NXFRAT, BASFRQ), two title lines and the
RATING records, then the bus section. The exporter writes the CIM names, so the two title lines are blank: a case's
titles have no CIM home.

In [9]:
text = written.read_text().splitlines()
print("\n".join(text[:3]))
bus_section = next(index for index, line in enumerate(text) if line.endswith("BEGIN BUS DATA"))
print("\n".join(text[bus_section:bus_section + 4]))

0,100.00,34,0,0,60.00


0 / END OF SYSTEM-WIDE DATA, BEGIN BUS DATA
1,'BUS 1       ',138.0000,3,1,1,,1.06000,0.0000,1.10000,0.90000,1.10000,0.90000
2,'BUS 2       ',138.0000,2,1,1,,1.04500,-4.9826,1.10000,0.90000,1.10000,0.90000
3,'BUS 3       ',138.0000,2,1,1,,1.01000,-12.7250,1.10000,0.90000,1.10000,0.90000


The written file reads back through `Raw34Reader`, with the same records. `cim_to_raw(network, path)` is the export
and the write in one call, and `raw_to_cim` is the import:

In [10]:
again = Raw34Reader(written).read_all()
print(again.count(schema.BusRecord), again.count(schema.BranchRecord), again.count(schema.TransformerRecord))

14 17 3


```python
from cimhub_raw.exporter.cim_to_raw import cim_to_raw

network = raw_to_cim(Raw34Reader(case).read_all(), name="IEEE14")
cim_to_raw(network, "out/IEEE14.raw")
```

## Next

- [Import](import.md): reader and importer options, the artifact files and what raises.
- [Export](export.md): bus numbers and IDs, the AREA trace, ratings and what raises.
- [Round trip](roundtrip.md): what comes back unchanged and what does not.
- [What converts](coverage.md): the RAW tables that import and export.